# Goal in life


- ngl i just wanna find a cool evalution function
- sum stuff i wanna try out not limited to but hopedully incliude


1) simple NN 
2) tranfomermer maybe
3) mcts

#Challenge 1:
- we need chess game data
- we needa format it
- i have like 4 and a half baraincells


In [ ]:

# load up everything
# we will start with simple ANN

import chess
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset


num_samples = 9000
raw_chess_dataset = load_dataset("Lichess/standard-chess-games", split="train", streaming=True)
chess_dataset = raw_chess_dataset.take(num_samples)


In [ ]:
import re

chess_df = pd.DataFrame(chess_dataset)

# first we get the stuff i think i want

games_df = chess_df[['Event', 'Result', 'Termination', 'movetext']]

# mapping there data into cleaner stuff
# rn i took a few extra features for potioental foture exploration


def parse_moves(movetext):
    results = {"1-0", "0-1", "1/2-1/2", "*"}
    movetext = re.sub(r"\{[^}]*\}", " ", movetext)   
    movetext = re.sub(r"\([^)]*\)", " ", movetext)   
    tokens = movetext.split()
    return [t.rstrip("!?") for t in tokens           
            if not t.endswith(".")                    
            and not t.startswith("$")                 
            and t not in results]


outcome_to_color = {
    "1-0" : "White",
    "0-1" : "Black",
    "1/2-1/2": "Draw"
}


rows = []
for _, row in games_df.iterrows():
    moves = row['movetext']
    all_moves = parse_moves(moves)
    outcome = row['Result']
    game_type = row['Event']
    end_type = row['Termination']
    rows.append(
        {
            "moves": all_moves, 
            "outcome": outcome_to_color[outcome],
            "game_type": game_type,
            "end_type": end_type
            })

X = pd.DataFrame(rows)

# rn we we games as a whole to an outcome, we needa map out each state within a game

def game_to_states(game_row):
    board_states = []
    board = chess.Board()
    num_moves = len(game_row['moves'])
    for index, move in enumerate(game_row['moves']):
        board.push_san(move)
        board_state = board.copy()
        board_states.append(board_state)
    return board_states


X['states'] = [game_to_states(row) for _, row in X.iterrows()]

columns = ['states', 'outcome']
true_x = X[columns]
true_x.head()

In [ ]:
#tensors requore numbers so we must map everything to numbers

# map board states to numbers (pieces to number then position to numbers)
# map outcomes to numbers

outcome_to_number = {
    "White": 1,
    "Black": -1,
    "Draw": 0
}

# originally i was doing satnadard 64 positons where each positojn had a number 
# 1 pawn, 2 horse, 3 biship .,,, 6 king 
# ig that was unstable so we moved to onehot

piece_order = [
    chess.Piece(chess.PAWN,   chess.WHITE), chess.Piece(chess.KNIGHT, chess.WHITE),
    chess.Piece(chess.BISHOP, chess.WHITE), chess.Piece(chess.ROOK,   chess.WHITE),
    chess.Piece(chess.QUEEN,  chess.WHITE), chess.Piece(chess.KING,   chess.WHITE),
    chess.Piece(chess.PAWN,   chess.BLACK), chess.Piece(chess.KNIGHT, chess.BLACK),
    chess.Piece(chess.BISHOP, chess.BLACK), chess.Piece(chess.ROOK,   chess.BLACK),
    chess.Piece(chess.QUEEN,  chess.BLACK), chess.Piece(chess.KING,   chess.BLACK),
]

piece_to_onehot = {piece: np.eye(12, dtype=np.float32)[index] for index, piece in enumerate(piece_order)}
# pawn W = [1 0 0 0 0 0 0 0 0 0  .. 0] king B [0 0 0 0 0 0 0 0 0 0  .. 1]


color_to_turn = {
    chess.WHITE: 1,
    chess.BLACK: -1
}

def game_to_numerical_rep(game):
    game_tensor = []
    for state in game:
        state_tensor = np.zeros(769, dtype=np.float32)
        for square, piece in state.piece_map().items():
            state_tensor[square * 12 : (square + 1) * 12] = piece_to_onehot[piece]
        state_tensor[768] = color_to_turn[state.turn]
        game_tensor.append(state_tensor)
    return game_tensor


true_x['numerical game rep'] = true_x['states'].apply(lambda x: game_to_numerical_rep(x))
true_x['outcome'] = true_x['outcome'].apply(lambda x: outcome_to_number[x])

""" 
so losng story short, keeping otocme for all states within a game is way to noisy,
so what mans thught is discoutned reward to give more accraue rep and giver
stronger opkenin to endgame non noisy staates 

intuitvely we with standsr aprpach same state can map to win or loss (imagine aerly board)
but with temoal discounts, those matter less 
"""

def discounted_labels(num_states, outcome, gamma):
    #One label per state: the last state
    labels = []
    for i in range(num_states):
        steps_from_end = num_states - 1 - i
        labels.append(outcome * gamma ** steps_from_end)
    return labels

gamma = 0.83
true_x['label'] = [
    discounted_labels(len(game), outcome, gamma)
    for game, outcome in zip(true_x['numerical game rep'], true_x['outcome'])
]


X_states = true_x[['numerical game rep', 'label']]

## NEW SHUFFEL
X_states = X_states.sample(frac=1, random_state=42).reset_index(drop=True)
##


X_states = pd.DataFrame(X_states)

X_states = X_states.explode(['numerical game rep', 'label'])
chess_data = X_states.to_numpy()
chess_data


In [ ]:
print(chess_data.shape)

X = np.stack(chess_data[:, 0])     
Y = chess_data[:, 1].astype(np.float32)  

print(X.shape, X.dtype)
print(Y.shape, Y.dtype)

cutoff = int(0.8 * len(X))

X_train = torch.tensor(X[:cutoff])
X_val   = torch.tensor(X[cutoff:])

Y_train = torch.tensor(Y[:cutoff])
Y_val   = torch.tensor(Y[cutoff:])

print(X_train.shape, Y_train.shape)
print(X_val.shape, Y_val.shape)

In [ ]:
X_train[0].shape

In [ ]:
import torch.nn as nn 
import torch.nn.functional as F

class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()
        self.W0 = nn.Linear(769, 512)
        self.W1 = nn.Linear(512, 256) 
        self.W2 = nn.Linear(256, 128)
        self.W3 = nn.Linear(128, 64)
        self.W4 = nn.Linear(64, 1)

    def forward(self, x):
        x0 =   F.leaky_relu(self.W0(x))
        x1 = F.leaky_relu(self.W1(x0))
        x2 = F.leaky_relu(self.W2(x1))
        x3 = F.leaky_relu(self.W3(x2))
        y_hat = torch.tanh(self.W4(x3))
        return y_hat

throwaya_model = ANN()
print(sum(p.numel() for p in throwaya_model.parameters()))   # ~58,113
print(throwaya_model(X_train[:5]).shape)                      # (5, 1)

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
# i should prolly hsiffle the traning and validaiotn set tohwrsie dependent events messup the porgrma
# well batch in advacne

def train(model, X_train, Y_train, X_val, Y_val, epochs, batch_size, lr, patience):
    global_train_error = []
    global_val_error = []
    train_dataset = TensorDataset(X_train, Y_train)
    val_dataset = TensorDataset(X_val, Y_val)
    
    
    
    mse = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    train_loader = DataLoader(
        dataset=train_dataset,
        batch_size=batch_size,
        shuffle=True
    )
    val_loader = DataLoader(
        dataset=val_dataset,
        batch_size=batch_size,
        shuffle=True
    )

    best_val = float("inf")
    epochs_without_improvement = 0

    for i in range(epochs):
        epoch_train_errors = []
        epoch_val_errors = []

        model.train()
        for batch in train_loader:
            x_true, y_true = batch
            optimizer.zero_grad() 
            y_hat = model(x_true).squeeze(-1)
            train_loss = mse(y_hat, y_true)
            epoch_train_errors.append(train_loss.item())
            train_loss.backward()
            optimizer.step()
        

        model.eval()
        with torch.no_grad():
            for batch in val_loader:
                x_true, y_true = batch
                y_hat = model(x_true).squeeze(-1)
                val_loss = mse(y_hat, y_true)
                epoch_val_errors.append(val_loss.item())
        
        train_mean = sum(epoch_train_errors) / len(epoch_train_errors)
        val_mean = sum(epoch_val_errors) / len(epoch_val_errors)

        global_train_error.append(train_mean)
        global_val_error.append(val_mean)

        # early stopping bookkeeping
        if val_mean < best_val:
            best_val = val_mean
            epochs_without_improvement = 0
            marker = " *"
        else:
            epochs_without_improvement += 1
            marker = ""

        print(f"epoch {i+1}/{epochs} | train {sum(epoch_train_errors)/len(epoch_train_errors):.4f} | val {sum(epoch_val_errors)/len(epoch_val_errors):.4f}")
        
        if epochs_without_improvement >= patience:
            print(f"early stop: no val improvement for {patience} epochs (best val {best_val:.4f})")
            break

    
    return global_train_error, global_val_error

In [ ]:
model = ANN()  
train_errors, val_errors = train(model, X_train, Y_train, X_val, Y_val, 10, 32, 1e-7, 3)

In [ ]:
print((Y_val ** 2).mean())                       # always predict 0
print(((Y_val - Y_train.mean()) ** 2).mean())    # always predict train mean

In [ ]:
torch.save(model.state_dict(), "chess_ann.pt")